**Github repository:** https://github.com/NicholasBorch/ComSocSci-Assignments

**Group Members:** Nicholas Borch (s234841) and Robin Braagaard (s234856)

**Distribution of Contribution:** All group members contributed equally in the planning and execution of solutions and implementation of code for the assignment tasks.

------

# 02467 Computational social science - Assignment 1

------

### Python libraries used in assigment

In [ ]:
### Importing libraries
from pathlib import Path
import pandas as pd
import re
import unicodedata
import time
from typing import Dict, Any, List, Optional, Tuple, Set
import numpy as np
import requests
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from unidecode import unidecode
from rapidfuzz import fuzz
from joblib import Parallel, delayed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from contextlib import contextmanager
import json

----

## Part 1: Ready Made vs Custom Made Data

> **Exercise: Ready made data vs Custom made data** In this exercise, I want to make sure you have understood they key points of my lecture and the reading. 

> 1. What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book __(answer in max 150 words)__.

#### Answer for P1Q1:

**Centola (Custom-made data):**

**Pros:** Strong causal clarity by having the network differences as the only causal variable. Experiment is designed explicitly around the research question. Participants self-opted into the platform following advertisements on health websites, not much suggests experiment awareness, suggesting a degree of non-reactivity.

**Cons:** Small scale (1,528 participants). Signing up to a forum is “low-cost”, genuine engagement is not captured; some may have signed up merely due to email annoyance. Study is unrepresentative, limited to internet users. Difficult to replicate as the platform is inaccessible to others.

**Nicolaides (Ready-made data):**

**Pros:** Massive scale (1.1M users, 350M+ km). Always-on continuous tracking over 5 years enables observation of emerging patterns. Based on real behaviour, not self-reports.

**Cons:** Non-representative, focusing on fitness-app users, probably an atypically health-motivated and competitive demographic, possibly exercising to beat friends rather than due to contagion. Algorithmic confounding likely, as the platform may artificially promote friend activity.



> 2. How do you think these differences can influence the interpretation of the results in each study? __(answer in max 150 words)__

#### Answer for P1Q2:

**Centola:** The small scale and artificial setting limit the generalisability of the experiment, results may not hold in larger, real-world networks. The low-cost behaviour that is being analyzed means we cannot conclude that clustered networks would similarly spread larger behavioural changes. However, the experimental control means the conclusions about network activity presented are highly credible within the study's scope.

**Nicolaides:** The massive scale and real-world setting strengthen generalisation. However, the non-representative sample of fitness-app users means results may not generalise too well to the rest of the population, which casts a few doubts on what the study finds the impact of contagion to be. Algorithmic confounding introduces uncertainty about whether observed contagion reflects genuine social influence or is being artificially pushed and constructed by the platform motivation. The reliance on behavioural data without further context means we cannot completely decide whether motivation or competitive pressure drives the observed behaviour.

------

## Pre Part 2 (Useful info and code from week 1):

Collecting the author names from the International Conference in Computational Social Science 2025.

Website URL: https://ic2s2-2025.org/program/

In [ ]:
# Output folder for Assignment 1 datasets
A1_DATA_DIR = Path("A1_data")
A1_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Source CSV for Web-scraping the list of participants to the International Conference in Computational Social Science (found via Network tab in browser dev tools)
IC2S2_SCHEDULE_CSV_URL = "https://ic2s2-2025.org/files/ic2s2_2025_schedule_v5.csv"

In [ ]:
# Setting up a function to canonicalize names, so that any spelling errors or differences in formatting 
# can be standardized for easier comparison and analysis and to avoid duplicates of the same person 
# being treated as different individuals.
def canon(name: str) -> str:
    """
    Canonical form of a person name for de-duplication:
    - lower-case
    - remove accents
    - transliterate remaining non-ASCII (ß->ss, ø->o, curly quotes->')
    - remove punctuation
    - collapse whitespace
    """
    if name is None:
        return ""
    name = str(name).strip().lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = unidecode(name)
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name.strip()

In [ ]:
### Loading IC2S2 schedule CSV
df_schedule = pd.read_csv(IC2S2_SCHEDULE_CSV_URL, storage_options={"User-Agent": "Mozilla/5.0"})
print("Rows:", len(df_schedule))
df_schedule.head()

In [ ]:
### Extracting authors and chairs into one raw name series
authors = df_schedule["author"].dropna().astype(str).str.split(",").explode().str.strip()
authors = authors[authors.ne("")]

chairs = df_schedule["chair"].dropna().astype(str).str.split(",").explode().str.strip()
chairs = chairs[chairs.ne("")]

# Combining the two lists
names_raw = pd.concat([authors, chairs], ignore_index=True)

print("Raw author entries:", len(authors))
print("Raw chair entries :", len(chairs))
print("Raw combined entries:", len(names_raw))
print("Unique (raw, exact string):", names_raw.nunique())
names_raw.head(10)

In [ ]:
# Creating a DataFrame that maps each raw name collected from the website 
# to its canonical version in order to standardize names and find 
# duplicates that may appear with slightly different spellings (e.g., accents, 
# punctuation differences, extra spaces). 
# Thereafter, counting how many unique canonical names exist (true unique researchers).
# and identify canonical names that correspond to multiple raw spellings.
names_df = pd.DataFrame({"name_raw": names_raw})
names_df["name_canon"] = names_df["name_raw"].map(canon)
print("Unique canonical names:", names_df["name_canon"].nunique())

# Canonical names that map to multiple raw spellings
variants = (names_df.groupby("name_canon")["name_raw"].unique().reset_index())
variants["n_variants"] = variants["name_raw"].apply(len)
variants_multi = variants[variants["n_variants"] > 1].sort_values("n_variants", ascending=False)

print("Canonical names with >1 raw variant:", len(variants_multi))
variants_multi

In [ ]:
# Setting up the final list of unique IC2S2 researchers.
# Due to multiple raw spellings that map to the same canonical name,
# raw names are sorted alphabetically and only one representative
# raw name per canonical version is kept.
unique_researchers = (names_df.sort_values("name_raw").drop_duplicates("name_canon")["name_raw"].reset_index(drop=True))

out_unique = A1_DATA_DIR / "D0_unique_researchers.csv"
unique_researchers.to_csv(out_unique, index=False, header=["name"])

print("Unique researchers:", len(unique_researchers))
unique_researchers.head(20)

In [ ]:
# In Week 2 (Part 2) text embeddings are used, hence a useful "text profile" is prepared for each person.

# For every row in the IC2S2 schedule (a talk/poster/etc.), a short document consisting of "title. abstract" is created.
# That document is then assigned to every author and chair listed on that row.

# This results in one text blob per researcher that summarizes what they contributed to at IC2S2 2025,
# and is used to find correct author ID in Part 2 of the assignment.
df_temp = df_schedule[["title", "abstract", "author", "chair"]].copy()
for col in ["title", "abstract", "author", "chair"]:
    df_temp[col] = df_temp[col].fillna("").astype(str)

rows = []
for _, r in df_temp.iterrows():
    paper_text = (r["title"].strip() + ". " + r["abstract"].strip()).strip()
    people = []
    people += [x.strip() for x in r["author"].split(",") if x.strip()]
    people += [x.strip() for x in r["chair"].split(",") if x.strip()]
    for person in people:
        rows.append({"name_raw": person, "name_canon": canon(person), "paper_text": paper_text})

df_people_text_long = pd.DataFrame(rows)

# Collecting all paper_text per person
df_people_text = (df_people_text_long.groupby("name_canon")["paper_text"].apply(lambda s: " ".join([t for t in s.tolist() if t])).reset_index().rename(columns={"paper_text": "conference_person_text"}))

out_person_text = A1_DATA_DIR / "conference_person_text.csv"
df_people_text.to_csv(out_person_text, index=False)

print("People with text:", len(df_people_text))
df_people_text.head()